In [1]:
import sys
import time
from pathlib import Path

import duckdb

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import build_feature_matrix

DB_PATH = ROOT / "data" / "home_credit.db"
conn = duckdb.connect(str(DB_PATH))

t0 = time.perf_counter()
df = build_feature_matrix(conn)
elapsed = time.perf_counter() - t0
print(f"First load (build_feature_matrix): {elapsed:.3f} s")
print(df.shape)

First load (build_feature_matrix): 2.800 s
(307511, 143)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.metrics import (
    binary_classifier_metrics,
    format_metrics_lines,
    plot_roc_pr_curves,
)

# Base features from `df`: exclude label and applicant id
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]

cat_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True),
        ),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
)

logit = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                solver="saga",
                random_state=42,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logit.fit(X_train, y_train)
y_score = logit.predict_proba(X_test)[:, 1]
y_pred = logit.predict(X_test)

test_metrics = binary_classifier_metrics(y_test, y_score, y_pred=y_pred)
print("Logistic regression — test set")
print(format_metrics_lines(test_metrics))

In [ ]:
fig = plot_roc_pr_curves(y_test, y_score, title_prefix="Test — ")